# Explore: Six Chapter 2 simulations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-chapter-2-simulations.ipynb)

**Chapter 2 · Beginner · six simulated samples**

These teaching datasets are generated in code, not collected from real students. We know the true treatment effects because we choose them. The six scenarios show how selection and confounding can change a simple treated-versus-untreated comparison.

Run the cells from top to bottom. You will use `pandas` to inspect tables, summaries, and plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def logistic(x):
    return 1 / (1 + np.exp(-x))

def make_scenario(seed, mode):
    np.random.seed(seed)
    n = 1000
    ability = np.random.normal(50, 10, n)
    income = np.random.normal(60000, 20000, n)
    motivation = np.random.normal(5, 2, n)
    if mode == "randomized_no_effect":
        prob = np.full(n, 0.5)
        effect = 0
        earnings = None
    elif mode == "randomized_positive_effect":
        prob = np.full(n, 0.5)
        effect = 5000
        earnings = None
    elif mode == "ability_negative_selection":
        prob = logistic((ability - 50) / 2)
        effect = 5000
        earnings = None
    elif mode == "ability_selection":
        prob = logistic((ability - 50) / 5)
        effect = 5000
        earnings = None
    elif mode == "income_confounded":
        prob = logistic((ability - 50) / 5 + (income - 60000) / 20000)
        effect = 5000
        earnings = None
    else:  # selection on ability, income, and motivation
        prob = logistic((ability - 50) / 5 + (income - 60000) / 20000 + (motivation - 5) / 2)
        effect = 5000
        earnings = None
    treated = np.random.binomial(1, prob, n)
    if mode == "randomized_no_effect":
        earnings = 30000 + 800 * ability + np.random.normal(0, 5000, n)
    elif mode == "randomized_positive_effect":
        earnings = 30000 + effect * treated + 800 * ability + np.random.normal(0, 5000, n)
    elif mode in ("ability_negative_selection", "ability_selection"):
        earnings = 30000 + effect * treated + 800 * ability + np.random.normal(0, 5000, n)
    elif mode == "income_confounded":
        earnings = 10000 + effect * treated + 800 * ability + .3 * income + np.random.normal(0, 5000, n)
    else:
        earnings = 10000 + effect * treated + 800 * ability + .3 * income + 2000 * motivation + np.random.normal(0, 5000, n)
    return pd.DataFrame({"treated": treated, "earnings": earnings, "ability": ability, "income": income, "motivation": motivation})

scenarios = {
    "1. Baseline selection on ability": make_scenario(42, "ability_selection"),
    "2. Random assignment, no effect": make_scenario(42, "randomized_no_effect"),
    "3. Stronger ability selection": make_scenario(42, "ability_negative_selection"),
    "4. Add family income confounding": make_scenario(42, "income_confounded"),
    "5. Random assignment, positive effect": make_scenario(42, "randomized_positive_effect"),
    "6. Select on ability, income, motivation": make_scenario(42, "all_confounders"),
}
print("Created", len(scenarios), "scenario dataframes")

## 1. Inspect one scenario

The rows are simulated students. `treated` is 1 for private school attendance and 0 otherwise. `earnings` is the outcome. `describe()` summarizes numeric columns.

In [ ]:
scenarios["1. Baseline selection on ability"].head()

In [ ]:
scenarios["1. Baseline selection on ability"].describe().T

## 2. Compare average earnings

Calculate treated and untreated averages in each scenario. In these simulations, random assignment protects the comparison from selection, while treatment effects and confounding differ by scenario.

In [ ]:
rows = []
for label, data in scenarios.items():
    means = data.groupby("treated")["earnings"].mean()
    rows.append({"Scenario": label, "Untreated mean": means.loc[0], "Treated mean": means.loc[1], "Difference": means.loc[1] - means.loc[0]})
comparison = pd.DataFrame(rows).set_index("Scenario")
comparison.round(1)

## 3. Plot a comparison

Look at how the raw mean difference changes across the scenarios. The true treatment effect is zero in scenario 2 and $5,000 in scenarios 3–6; scenario 1 has no treatment effect either. Sampling and selection can make the raw differences differ from those values.

In [ ]:
comparison["Difference"].plot(kind="barh", figsize=(9, 5), title="Treated minus untreated mean earnings")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Difference in simulated earnings")
plt.tight_layout()
plt.show()

## 4. Explore the role of ability

**Try it:** compare average ability across treated and untreated groups in each scenario. What happens under random assignment?

**Hint:** use `groupby("treated")["ability"].mean()`.

**Optional solution:**

In [ ]:
pd.DataFrame({label: data.groupby("treated")["ability"].mean() for label, data in scenarios.items()}).T.round(2)

## 5. Interpret carefully

- Which raw mean differences are closest to the known treatment effect?
- In which scenarios are treated students different in ability or income before comparing earnings?
- How does random assignment change the comparison?

Chapter 2 uses simulations to explain omitted variable bias and regression adjustment. These generated examples illustrate ideas under chosen equations; real observational data do not reveal the true effect automatically. An exploratory group difference shows an association and cannot, by itself, establish causality.